In [8]:
# Cell 2: Imports
# %%
import os
import random
import joblib
import json
from pathlib import Path
from typing import Tuple, Any, Dict, List

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tabulate import tabulate

# Try imports that may not be installed in all environments
try:
    import lightgbm as lgb
except Exception:
    lgb = None
try:
    import xgboost as xgb
except Exception:
    xgb = None
try:
    from tabpfn import TabPFNClassifier
except Exception:
    TabPFNClassifier = None
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
except Exception:
    torch = None

from datahandles import FewShotDataset


In [9]:
# Cell 3: Placeholder FewShotDataset
# %%
"""
This cell provides a minimal placeholder/fallback FewShotDataset class so the notebook runs end-to-end
Replace this with your real FewShotDataset creator (paste it here or import it) — the rest of the notebook
will try to automatically use the class you provide.

The placeholder exposes a simple API:
  - instantiate as FewShotDataset(X, y, test_size=0.2, val_size=0.1, random_seed=42)
  - attributes: X_train, y_train, X_val, y_val, X_test, y_test

If you already have a FewShotDataset with a different API, either paste it here or modify
the `get_splits_from_fewshot` helper below to adapt to your API.
"""

class FewShotDatasetFallback:
    def __init__(self, X, y, test_size=0.2, val_size=0.1, random_seed=42):
        self.random_seed = random_seed
        self.X = np.asarray(X)
        self.y = np.asarray(y)
        self.test_size = test_size
        self.val_size = val_size
        self._create_splits()

    def _create_splits(self):
        rs = self.random_seed
        X_trainval, self.X_test, y_trainval, self.y_test = train_test_split(
            self.X, self.y, test_size=self.test_size, random_state=rs, stratify=self.y
        )
        # Now split train/val from trainval
        rel_val = self.val_size / (1.0 - self.test_size)
        self.X_train, self.X_val, self.y_train, self.y_val = train_test_split(
            X_trainval, y_trainval, test_size=rel_val, random_state=rs + 1, stratify=y_trainval
        )


# To allow user to override the class name FewShotDataset, try to import or refer to a variable
# named FewShotDataset in the notebook globals. If present, we'll prefer it; otherwise use fallback.

# try:
#     FewShotDataset  # type: ignore
#     print("Successfully imported FewShotDataset")
# except NameError:
#     FewShotDataset = FewShotDatasetFallback



In [10]:
# Cell 4: Helper to robustly extract splits from the fewshot object
# %%

def get_splits_from_fewshot(ds_name, n_shots, random_seed) -> Dict[str, np.ndarray]:
    """Return X_train, y_train, X_val, y_val, X_test, y_test.
    Tries multiple common attribute names so it works with various FewShotDataset implementations.
    """    
    train_ds = FewShotDataset(
        dataset_names=[ds_name],
        data_root="./data",
        split="train",
        split_size=1,
        n_shots=n_shots,
        n_queries=n_shots,
        queries_same_as_shots=True,
        max_n_features=None,
        balance_labels=True,
        col_permutation=False,
        shuffle=True,
        debug=False,
        random_seed=random_seed,
        shots_with_labels=False
    )

    # val_ds = FewShotDataset(
    #     dataset_names=[ds_name],
    #     data_root="./data",
    #     split="val",
    #     split_size=1,
    #     n_shots=100,
    #     n_queries=100,
    #     queries_same_as_shots=True,
    #     max_n_features=None,
    #     balance_labels=True,
    #     col_permutation=False,
    #     shuffle=True,
    #     debug=False,
    #     random_seed=random_seed,
    #     shots_with_labels=False
    # )
    val_ds = train_ds

    return {
        "X_train": train_ds[0]["queries_x"], 
        "y_train": train_ds[0]["queries_y"], 
        "X_val": val_ds[0]["queries_x"], 
        "y_val": val_ds[0]["queries_y"],
    }



In [11]:
# Cell 5: Model utilities and MLP definition
# %%

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


class TorchMLP(nn.Module):
    def __init__(self, input_dim: int, n_layers: int, width: int, output_dim: int = 1):
        super().__init__()
        layers = []
        in_dim = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.ReLU())
            in_dim = width
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_torch_mlp(X_train, y_train, X_val, y_val, mlp_config, device='cpu', epochs=30, batch_size=32, lr=1e-3, seed=0, ckpt_path=None):
    if torch is None:
        raise RuntimeError("PyTorch not installed. Install torch to train the MLP.")
    set_global_seed(seed)
    model = TorchMLP(input_dim=X_train.shape[1], n_layers=mlp_config[0], width=mlp_config[1]).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=lr)

    train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
    val_ds = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    best_val_auc = -1.0
    best_epoch = -1
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
        # Validate
        model.eval()
        ys, preds = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy()
                preds.append(probs)
                ys.append(yb.numpy())
        ys = np.concatenate(ys)
        preds = np.concatenate(preds)
        try:
            val_auc = roc_auc_score(ys, preds)
        except Exception:
            val_auc = float('nan')
        if ckpt_path is not None:
            # Save checkpoint each epoch (or only best if you prefer)
            ckpt_file = ckpt_path / f"mlp_epoch{ep}.pt"
            torch.save({
                'epoch': ep,
                'model_state_dict': model.state_dict(),
                'opt_state_dict': opt.state_dict(),
                'val_auc': val_auc,
                'seed': seed,
                'mlp_config': mlp_config
            }, str(ckpt_file))
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = ep
    return model, best_val_auc, best_epoch



In [12]:
# Cell 6: High-level training orchestration
# %%

def train_and_save_all_models(
    ds_name: str,
    n_shots: int,
    seeds: List[int],
    mlp_arch: Tuple[int, int],
    out_dir: str = "classical_checkpoints",
    mlp_epochs: int = 30,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    results = []

    for seed in seeds:
        print(f"\n=== Running seed {seed} ===")
        set_global_seed(seed)

        data = get_splits_from_fewshot(ds_name=ds_name,
                                   n_shots=n_shots,
                                   random_seed=seed)
        
        X_train = np.array(data["X_train"], dtype=np.float64)
        y_train = np.array(data["y_train"], dtype=np.long)
        X_val = np.array(data["X_val"], dtype=np.float64)
        y_val = np.array(data["y_val"], dtype=np.long)

        print(f"Train set size: {len(X_train)}, Validation set size: {len(X_val)}")

        print(type(X_train))

        seed_dir = out_dir / f"seed_{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)

        # 1) Logistic Regression
        print("Training Logistic Regression...")
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))
        joblib.dump(clf, seed_dir / "logistic.pkl")

        # 2) LightGBM
        if lgb is not None:
            print("Training LightGBM...")
            lgb_train = lgb.Dataset(np.vstack([X_train, X_val]), label=np.concatenate([y_train, y_val]))
            params = {"objective": "binary", "metric": "auc", "verbosity": -1, "seed": seed}
            gbm = lgb.train(params, lgb_train, num_boost_round=100)
            gbm.save_model(str(seed_dir / "lightgbm.txt"))
        else:
            print("LightGBM not installed; skipping")

        # 3) XGBoost
        if xgb is not None:
            print("Training XGBoost...")
            dtrain = xgb.DMatrix(np.vstack([X_train, X_val]), label=np.concatenate([y_train, y_val]))
            param = {"objective": "binary:logistic", "eval_metric": "auc", "seed": seed}
            bst = xgb.train(param, dtrain, num_boost_round=100)
            bst.save_model(str(seed_dir / "xgboost.json"))
        else:
            print("XGBoost not installed; skipping")

        if TabPFNClassifier is not None:
            tabpfn = TabPFNClassifier(device='cpu')
            tabpfn.fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))
            joblib.dump(tabpfn, seed_dir / "tabpfn.pkl")

        # 4) MLP (PyTorch)
        if torch is not None:
            print("Training MLP (PyTorch)...")
            ckpt_subdir = seed_dir / "mlp_checkpoints"
            ckpt_subdir.mkdir(exist_ok=True)
            model, best_val_auc, best_epoch = train_torch_mlp(
                np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]),
                X_val, y_val, mlp_arch, device='cpu', epochs=mlp_epochs,
                batch_size=32, lr=1e-3, seed=seed, ckpt_path=ckpt_subdir
            )
            # Save final model as well
            final_path = seed_dir / "mlp_final.pt"
            torch.save({'model_state_dict': model.state_dict(), 'mlp_arch': mlp_arch}, str(final_path))
        else:
            print("PyTorch not installed; skipping MLP")

    print("\n=== Done training all seeds ===")



In [13]:
# Cell 7: Evaluation — load checkpoints and compute ROC-AUC on test set
# %%

def evaluate_checkpoints(checkpoint_root: str, X_test: np.ndarray, y_test: np.ndarray) -> pd.DataFrame:
    root = Path(checkpoint_root)
    rows = []
    for seed_dir in sorted(root.glob('seed_*')):
        seed = int(seed_dir.name.split('_')[-1])
        # Logistic
        log_path = seed_dir / 'logistic.pkl'
        if log_path.exists():
            clf = joblib.load(log_path)
            probs = clf.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, probs)
            rows.append({'model': 'LogisticRegression', 'seed': seed, 'roc_auc': auc})

        # LightGBM
        lgb_path = seed_dir / 'lightgbm.txt'
        if lgb_path.exists() and lgb is not None:
            gbm = lgb.Booster(model_file=str(lgb_path))
            probs = gbm.predict(X_test)
            auc = roc_auc_score(y_test, probs)
            rows.append({'model': 'LightGBM', 'seed': seed, 'roc_auc': auc})

        # XGBoost
        xgb_path = seed_dir / 'xgboost.json'
        if xgb_path.exists() and xgb is not None:
            bst = xgb.Booster()
            bst.load_model(str(xgb_path))
            dtest = xgb.DMatrix(X_test)
            probs = bst.predict(dtest)
            auc = roc_auc_score(y_test, probs)
            rows.append({'model': 'XGBoost', 'seed': seed, 'roc_auc': auc})

        # TabPFN
        tabpfn_path = seed_dir / 'tabpfn.pkl'
        if tabpfn_path.exists() and TabPFNClassifier is not None:
            tabpfn = joblib.load(tabpfn_path)
            probs = tabpfn.predict_proba(X_test)[:, 1]
            rows.append({'model': 'TabPFN', 'seed': seed, 'roc_auc': roc_auc_score(y_test, probs)})

        # MLP - attempt to load final or best checkpoint
        mlp_final = seed_dir / 'mlp_final.pt'
        ckpt_subdir = seed_dir / 'mlp_checkpoints'
        if (mlp_final.exists() or (ckpt_subdir.exists() and any(ckpt_subdir.glob('mlp_epoch*.pt')))) and torch is not None:
            # Prefer final
            if mlp_final.exists():
                data = torch.load(str(mlp_final), map_location='cpu')
                arch = data.get('mlp_arch', None)
                model = TorchMLP(input_dim=X_test.shape[1], n_layers=arch[0], width=arch[1])
                model.load_state_dict(data['model_state_dict'])
            else:
                # pick last epoch file
                epochs = sorted(ckpt_subdir.glob('mlp_epoch*.pt'))
                last = epochs[-1]
                data = torch.load(str(last), map_location='cpu')
                arch = data.get('mlp_config', None)
                model = TorchMLP(input_dim=X_test.shape[1], n_layers=arch[0], width=arch[1])
                model.load_state_dict(data['model_state_dict'])
            model.eval()
            with torch.no_grad():
                inp = torch.from_numpy(X_test).float()
                logits = model(inp)
                probs = torch.sigmoid(logits).numpy()
            auc = roc_auc_score(y_test, probs)
            rows.append({'model': 'MLP', 'seed': seed, 'roc_auc': auc})

    df = pd.DataFrame(rows)
    return df



In [ ]:
# Cell 8: Example usage end-to-end
# %%
if __name__ == '__main__':
    # User-configurable parameters
    ds_name = "calhousing"
    n_shots = 8
    seeds = [14, 26, 42, 58, 97]
    mlp_arch = (4, 10)  # (n_hidden_layers, width_of_hidden_layers)
    mlp_epochs = 100
    checkpoint_dir = f"checkpoints/{ds_name}/n{n_shots:0>2}"

    n_queries_dict = {"bank": 43211, "calhousing": 19640, "income": 44222}
    test_ds = FewShotDataset(
        dataset_names=[ds_name],
        data_root="./data",
        split="test",
        split_size=1,
        n_shots=n_queries_dict[ds_name],
        n_queries=n_queries_dict[ds_name],
        queries_same_as_shots=True,
        max_n_features=None,
        balance_labels=False,
        col_permutation=False,
        shuffle=True,
        debug=False,
        random_seed=True,
        shots_with_labels=False
    )

    X_test = np.array(test_ds[0]["queries_x"], dtype=np.float64)
    y_test = np.array(test_ds[0]["queries_y"], dtype=np.long)

    print(f"Test set size: {len(X_test)}")

    # Train & save
    train_and_save_all_models(ds_name=ds_name,
                              n_shots=n_shots,
                              seeds=seeds, 
                              mlp_arch=mlp_arch, 
                              out_dir=checkpoint_dir, 
                              mlp_epochs=mlp_epochs)

    # Evaluate
    df_results = evaluate_checkpoints(checkpoint_dir, X_test, y_test)

    # Save CSV and print nicely
    out_csv = Path(checkpoint_dir) / 'results.csv'
    df_results.to_csv(out_csv, index=False)

    # Aggregate and print table
    print('\n=== Results per model/seed ===')
    print(tabulate(df_results.sort_values(['model', 'seed']), headers='keys', tablefmt='github', showindex=False, floatfmt='.4f'))

    # Also print mean per model
    print('\n=== Mean ROC-AUC per model ===')
    print(tabulate(df_results.groupby('model')['roc_auc'].agg(['mean', 'std']).reset_index(), headers='keys', tablefmt='github', showindex=False, floatfmt='.4f'))

    print(f"\nSaved results to: {out_csv}")


/tmp/ipykernel_3823861/569517865.py:30: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X_test = np.array(test_ds[0]["queries_x"], dtype=np.float64)
/tmp/ipykernel_3823861/569517865.py:31: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_test = np.array(test_ds[0]["queries_y"], dtype=np.long)
/tmp/ipykernel_3823861/1235894223.py:25: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'co

Test set size: 19640

=== Running seed 14 ===
Train set size: 8, Validation set size: 8
<class 'numpy.ndarray'>
Training Logistic Regression...
Training LightGBM...
Training XGBoost...
